In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

train_df = pd.read_csv("Bal_train_dataset.csv")
test_df  = pd.read_csv("Bal_test_dataset.csv")

tfidf = TfidfVectorizer(
    max_features=30000,     # more features for 20k reviews
    ngram_range=(1, 3),     # include tri-grams
    min_df=3,               # ignore words that appear in <3 docs
    max_df=0.8,             # ignore overly common words
    sublinear_tf=True,      # log-scaling term frequency
    stop_words='english'    # remove stopwords
)

X_train = tfidf.fit_transform(train_df['Review'])
X_test  = tfidf.transform(test_df['Review'])
y_train = train_df['Rating']
y_test  = test_df['Rating']


In [25]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

# Convert predictions to nearest integer (1–5)
y_pred_round = [round(y) for y in y_pred]
acc = accuracy_score(y_test, y_pred_round)

print("Linear Regression Accuracy:", acc)


Linear Regression Accuracy: 0.2182


In [26]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Logistic Regression Accuracy: 0.4866

Classification Report:
               precision    recall  f1-score   support

           1       0.56      0.61      0.58      1000
           2       0.41      0.36      0.38      1000
           3       0.40      0.36      0.38      1000
           4       0.46      0.44      0.45      1000
           5       0.58      0.65      0.61      1000

    accuracy                           0.49      5000
   macro avg       0.48      0.49      0.48      5000
weighted avg       0.48      0.49      0.48      5000



In [27]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(C=10, class_weight='balanced')
svm_model.fit(X_train, y_train)
y_pred = svm_model.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


SVM Accuracy: 0.418

Classification Report:
               precision    recall  f1-score   support

           1       0.52      0.53      0.53      1000
           2       0.36      0.33      0.34      1000
           3       0.32      0.32      0.32      1000
           4       0.38      0.39      0.39      1000
           5       0.50      0.52      0.51      1000

    accuracy                           0.42      5000
   macro avg       0.42      0.42      0.42      5000
weighted avg       0.42      0.42      0.42      5000



In [28]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
y_pred = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Naive Bayes Accuracy: 0.4724

Classification Report:
               precision    recall  f1-score   support

           1       0.53      0.64      0.58      1000
           2       0.40      0.33      0.36      1000
           3       0.38      0.37      0.38      1000
           4       0.43      0.38      0.40      1000
           5       0.57      0.65      0.61      1000

    accuracy                           0.47      5000
   macro avg       0.46      0.47      0.47      5000
weighted avg       0.46      0.47      0.47      5000



In [29]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(max_depth=100, random_state=42)
dt_model.fit(X_train, y_train)
y_pred = dt_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Decision Tree Accuracy: 0.3258

Classification Report:
               precision    recall  f1-score   support

           1       0.41      0.42      0.41      1000
           2       0.26      0.25      0.25      1000
           3       0.28      0.26      0.27      1000
           4       0.29      0.31      0.30      1000
           5       0.38      0.39      0.38      1000

    accuracy                           0.33      5000
   macro avg       0.32      0.33      0.32      5000
weighted avg       0.32      0.33      0.32      5000



In [33]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# ===============================
# 1️⃣ Load Balanced Dataset
# ===============================
train_df = pd.read_csv("Bal_train_dataset.csv")
test_df = pd.read_csv("Bal_test_dataset.csv")

X_train, y_train = train_df["Review"], train_df["Rating"]
X_test, y_test = test_df["Review"], test_df["Rating"]

# ===============================
# 2️⃣ TF-IDF Feature Extraction (Optimized)
# ===============================
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 3),
    min_df=3,
    max_df=0.8,
    sublinear_tf=True,
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# ===============================
# 3️⃣ Fine-Tuning Models
# ===============================
results = {}

print("\n🔧 Fine-Tuning Selected Models (Balanced Dataset)\n")

# --- Logistic Regression ---
best_acc, best_params = 0, None
for c in [0.01, 0.1, 1, 5, 10]:
    model = LogisticRegression(C=c, max_iter=2000, solver='liblinear')
    model.fit(X_train_tfidf, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_tfidf))
    if acc > best_acc:
        best_acc, best_params = acc, c
results["Logistic Regression"] = (best_acc, f"C={best_params}")
print(f"✅ Logistic Regression: {best_acc:.4f} | Best C={best_params}")

# --- Linear SVM (LinearSVC) ---
best_acc, best_params = 0, None
for c in [0.01, 0.1, 1, 5, 10]:
    model = LinearSVC(C=c, class_weight='balanced', max_iter=5000)
    model.fit(X_train_tfidf, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_tfidf))
    if acc > best_acc:
        best_acc, best_params = acc, c
results["Linear SVM"] = (best_acc, f"C={best_params}")
print(f"✅ Linear SVM: {best_acc:.4f} | Best C={best_params}")

# --- Multinomial Naive Bayes ---
best_acc, best_params = 0, None
for alpha in [0.1, 0.3, 0.5, 1.0]:
    model = MultinomialNB(alpha=alpha)
    model.fit(X_train_tfidf, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_tfidf))
    if acc > best_acc:
        best_acc, best_params = acc, alpha
results["MultinomialNB"] = (best_acc, f"alpha={best_params}")
print(f"✅ MultinomialNB: {best_acc:.4f} | Best alpha={best_params}")

# ===============================
# 4️⃣ Summary
# ===============================
print("\n📊 Best Model Comparison (Balanced Dataset):\n")
for model, (acc, params) in results.items():
    print(f"{model:22s} | Accuracy: {acc:.4f} | Params: {params}")

best_model = max(results, key=lambda x: results[x][0])
print(f"\n🏆 Best Overall Model: {best_model} → Accuracy: {results[best_model][0]:.4f} ({results[best_model][1]})")



🔧 Fine-Tuning Selected Models (Balanced Dataset)

✅ Logistic Regression: 0.4926 | Best C=1
✅ Linear SVM: 0.4918 | Best C=0.1
✅ MultinomialNB: 0.4724 | Best alpha=1.0

📊 Best Model Comparison (Balanced Dataset):

Logistic Regression    | Accuracy: 0.4926 | Params: C=1
Linear SVM             | Accuracy: 0.4918 | Params: C=0.1
MultinomialNB          | Accuracy: 0.4724 | Params: alpha=1.0

🏆 Best Overall Model: Logistic Regression → Accuracy: 0.4926 (C=1)


In [34]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# =======================================
# 1️⃣ Load Dataset
# =======================================
train_df = pd.read_csv("Bal_train_dataset.csv")
test_df = pd.read_csv("Bal_test_dataset.csv")

X_train, y_train = train_df["Review"], train_df["Rating"]
X_test, y_test = test_df["Review"], test_df["Rating"]

# =======================================
# 2️⃣ Optimized TF-IDF Vectorization
# =======================================
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 3),
    min_df=3,
    max_df=0.8,
    sublinear_tf=True,
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# =======================================
# 3️⃣ Fine-Tuning Logistic Regression, LinearSVC, and MultinomialNB
# =======================================
results = {}

print("\n🔧 Fine-Tuning on BALANCED Dataset\n")

# --- Logistic Regression ---
print("Logistic Regression (varying C):")
best_acc, best_C = 0, None
for C in [0.01, 0.1, 1, 5, 10]:
    model = LogisticRegression(C=C, solver='liblinear', max_iter=2000)
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    print(f"  C={C:<5} → Accuracy: {acc:.4f}")
    if acc > best_acc:
        best_acc, best_C = acc, C
results["Logistic Regression"] = (best_acc, f"C={best_C}")

# --- Linear SVM ---
print("\nLinear SVM (varying C):")
best_acc, best_C = 0, None
for C in [0.01, 0.1, 1, 5, 10]:
    model = LinearSVC(C=C, class_weight='balanced', max_iter=5000)
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    print(f"  C={C:<5} → Accuracy: {acc:.4f}")
    if acc > best_acc:
        best_acc, best_C = acc, C
results["Linear SVM"] = (best_acc, f"C={best_C}")

# --- Multinomial Naive Bayes ---
print("\nMultinomial Naive Bayes (varying alpha):")
best_acc, best_alpha = 0, None
for alpha in [0.1, 0.3, 0.5, 1.0]:
    model = MultinomialNB(alpha=alpha)
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    print(f"  alpha={alpha:<3} → Accuracy: {acc:.4f}")
    if acc > best_acc:
        best_acc, best_alpha = acc, alpha
results["MultinomialNB"] = (best_acc, f"alpha={best_alpha}")

# =======================================
# 4️⃣ Summary
# =======================================
print("\n📊 Final Comparison (Balanced Dataset):\n")
for model, (acc, params) in results.items():
    print(f"{model:22s} | Accuracy: {acc:.4f} | Params: {params}")

best_model = max(results, key=lambda x: results[x][0])
print(f"\n🏆 Best Model: {best_model} → Accuracy: {results[best_model][0]:.4f} ({results[best_model][1]})")



🔧 Fine-Tuning on BALANCED Dataset

Logistic Regression (varying C):
  C=0.01  → Accuracy: 0.4614
  C=0.1   → Accuracy: 0.4758
  C=1     → Accuracy: 0.4926
  C=5     → Accuracy: 0.4676
  C=10    → Accuracy: 0.4614

Linear SVM (varying C):
  C=0.01  → Accuracy: 0.4732
  C=0.1   → Accuracy: 0.4918
  C=1     → Accuracy: 0.4556
  C=5     → Accuracy: 0.4296
  C=10    → Accuracy: 0.4180

Multinomial Naive Bayes (varying alpha):
  alpha=0.1 → Accuracy: 0.4512
  alpha=0.3 → Accuracy: 0.4582
  alpha=0.5 → Accuracy: 0.4654
  alpha=1.0 → Accuracy: 0.4724

📊 Final Comparison (Balanced Dataset):

Logistic Regression    | Accuracy: 0.4926 | Params: C=1
Linear SVM             | Accuracy: 0.4918 | Params: C=0.1
MultinomialNB          | Accuracy: 0.4724 | Params: alpha=1.0

🏆 Best Model: Logistic Regression → Accuracy: 0.4926 (C=1)


In [36]:
import pandas as pd
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from tabulate import tabulate  # 👈 for table view

# -----------------------------
# 1️⃣ Load datasets
# -----------------------------
train_df = pd.read_csv("Bal_train_dataset.csv")
test_df  = pd.read_csv("Bal_test_dataset.csv")

X_train, y_train = train_df["Review"], train_df["Rating"]
X_test, y_test   = test_df["Review"], test_df["Rating"]

# -----------------------------
# 2️⃣ TF-IDF Vectorization (same params as tuned)
# -----------------------------
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,3),
    min_df=3,
    max_df=0.8,
    sublinear_tf=True,
    stop_words='english'
)
X_train_tfidf = tfidf.fit_transform(X_train)

# -----------------------------
# 3️⃣ Final Logistic Regression (C=1)
# -----------------------------
final_model = LogisticRegression(C=1, solver='liblinear', max_iter=2000)
final_model.fit(X_train_tfidf, y_train)

# -----------------------------
# 4️⃣ Select random test reviews (5 per rating)
# -----------------------------
ratings = [1, 2, 3, 4, 5]
sample_reviews = []

for r in ratings:
    subset = test_df[test_df['Rating'] == r]
    if len(subset) >= 5:
        review_samples = subset.sample(n=5, random_state=random.randint(0, 1000))
    else:
        review_samples = subset  # if fewer than 5 exist
    for _, row in review_samples.iterrows():
        sample_reviews.append((r, row['Review']))

# -----------------------------
# 5️⃣ Predict and store results
# -----------------------------
results = []

for idx, (true_rating, review_text) in enumerate(sample_reviews, start=1):
    X_vec = tfidf.transform([review_text])
    pred_rating = final_model.predict(X_vec)[0]
    results.append({
        "S.No": idx,
        "Actual Rating": true_rating,
        "Predicted Rating": pred_rating,
        "Review (First 200 chars)": review_text[:200] + ("..." if len(review_text) > 200 else "")
    })

# -----------------------------
# 6️⃣ Display as a formatted table
# -----------------------------
results_df = pd.DataFrame(results)

print("\n📊 Model Prediction Test on Random Reviews (Balanced Dataset):\n")
print(tabulate(results_df, headers="keys", tablefmt="grid", showindex=False))



📊 Model Prediction Test on Random Reviews (Balanced Dataset):

+--------+-----------------+--------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|   S.No |   Actual Rating |   Predicted Rating | Review (First 200 chars)                                                                                                                                                                                    |
+========+=================+====================+=============================================================================================================================================================================================================+
|      1 |               1 |                  3 | This product tastes good, but you can buy a container of each type for $9 at Wegman's. If you get it from this place, 

In [41]:
import joblib

# Save trained model and vectorizer
joblib.dump(final_model, "Model_A.pkl")
joblib.dump(tfidf, "tfidf_bal.pkl")

print("✅ Model_A (Balanced Logistic Regression) and tfidf_bal.pkl saved successfully!")


✅ Model_A (Balanced Logistic Regression) and tfidf_bal.pkl saved successfully!


In [42]:
from google.colab import files

files.download("Model_A.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
files.download("tfidf_bal.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>